In [0]:
%run "../SetUp/setup" 

### Access Azure Data Lake using Service Principal
**Steps to follow:**
1. Register Azure AD Application/ Service Principal
2. Generate a secret/ password for the application
3. Set spark config with App/ Client Id, Directory/ Tenant Id & Secret
4. Assign role "Storage Blob Data Contributor" to the Data Lake

path,name,size,modificationTime
abfss://raw@forrmulaa1dl.dfs.core.windows.net/2021-03-21/,2021-03-21/,0,1768733801000
abfss://raw@forrmulaa1dl.dfs.core.windows.net/2021-03-28/,2021-03-28/,0,1768733591000
abfss://raw@forrmulaa1dl.dfs.core.windows.net/2021-04-18/,2021-04-18/,0,1768733721000


[FileInfo(path='abfss://demo@forrmulaa1dl.dfs.core.windows.net/__unitystorage/', name='__unitystorage/', size=0, modificationTime=1769227742000),
 FileInfo(path='abfss://demo@forrmulaa1dl.dfs.core.windows.net/drivers_convert_to_delta/', name='drivers_convert_to_delta/', size=0, modificationTime=1769359802000),
 FileInfo(path='abfss://demo@forrmulaa1dl.dfs.core.windows.net/drivers_convert_to_delta_new/', name='drivers_convert_to_delta_new/', size=0, modificationTime=1769360049000),
 FileInfo(path='abfss://demo@forrmulaa1dl.dfs.core.windows.net/results_external/', name='results_external/', size=0, modificationTime=1769228062000),
 FileInfo(path='abfss://demo@forrmulaa1dl.dfs.core.windows.net/results_partitioned/', name='results_partitioned/', size=0, modificationTime=1769228706000)]

In [0]:
%run "../Includes/configs" 

In [0]:
%run "../Includes/comm_func" 

**Produce Driver Standings**

In [0]:
dbutils.widgets.text("p_file_date", "")

In [0]:
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
from pyspark.sql.functions import sum, when, col, count, lit

In [0]:
race_results_df = spark.read.format("delta").load(f"{presentation_folder_path}/race_results")

In [0]:
driver_standings_df = race_results_df.groupBy("race_year", "driver_name", "driver_nationality").agg(sum("points").alias("total_points"), count(when(col("position") == 1, True)).alias("wins"))

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, desc, asc

In [0]:
drivers_rank_spec = Window.partitionBy("race_year").orderBy(desc("total_points"), desc("wins"))
final_df = driver_standings_df.withColumn("rank", rank().over(drivers_rank_spec)).withColumn("file_date", lit(v_file_date))

In [0]:
#final_df.write.mode("overwrite").parquet(f"{presentation_folder_path}/driver_standings")

In [0]:
final_df_dedup = final_df.dropDuplicates(
    ["race_year", "driver_name"]
)

In [0]:
final_df_dedup.write \
    .mode("append") \
    .format("delta") \
    .partitionBy("race_year") \
    .save(f"{presentation_folder_path}/driver_standings")

In [0]:
spark.read.format("delta").load(f"{presentation_folder_path}/driver_standings") \
    .select("file_date") \
    .distinct() \
    .show(truncate=False)

+----------+
|file_date |
+----------+
|2021-03-21|
|2021-04-18|
|2021-03-28|
+----------+



In [0]:
merge_condition = "tgt.driver_name = src.driver_name AND tgt.race_year = src.race_year"
merge_delta_data(final_df_dedup, 'f1_presentation', 'driver_standings', presentation_folder_path, merge_condition, 'race_year')

In [0]:
%sql
SELECT * from f1_presentation.driver_standings
ORDER BY race_year DESC;

race_year,driver_name,driver_nationality,total_points,wins,rank,file_date
2021,Antonio Giovinazzi,Italian,0.0,0,14,2021-04-18
2021,Carlos Sainz,Spanish,28.0,0,6,2021-04-18
2021,Charles Leclerc,Monegasque,40.0,0,4,2021-04-18
2021,Daniel Ricciardo,Australian,28.0,0,6,2021-04-18
2021,Esteban Ocon,French,4.0,0,11,2021-04-18
2021,Fernando Alonso,Spanish,2.0,0,13,2021-04-18
2021,George Russell,British,0.0,0,14,2021-04-18
2021,Kimi Räikkönen,Finnish,0.0,0,14,2021-04-18
2021,Lance Stroll,Canadian,10.0,0,10,2021-04-18
2021,Lando Norris,British,54.0,0,3,2021-04-18


In [0]:
%sql
SELECT race_year, COUNT(1) FROM f1_presentation.driver_standings
GROUP BY race_year
ORDER BY race_year DESC;

race_year,count(1)
2021,20
2020,23
2019,20
2018,20
2017,25
2016,24
2015,22
2014,24
2013,23
2012,25
